# Modelo de aprendizaje automático para brillo y cinemática residual de las Gemínidas

In [16]:
import hashlib
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    PolynomialFeatures,
    StandardScaler,
)


## Etapa 11. Preparación del modelado y diseño de la validación temporal

### Carga y auditoría de las bases de modelado

In [2]:
# Importar las dependencias y comprobar las rutas de los archivos de modelado

ruta_actual = Path.cwd().resolve()
raiz_proyecto = next((ruta for ruta in [ruta_actual, *ruta_actual.parents] if (ruta / "pyproject.toml").is_file()), None)

if raiz_proyecto is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto a partir del directorio actual.")

carpeta_modelado = raiz_proyecto / "data" / "processed" / "geminidas_modelado_v2"
ruta_entrenamiento = carpeta_modelado / "geminidas_modelado_entrenamiento_2019_2024_v2.parquet"
ruta_prueba = carpeta_modelado / "geminidas_modelado_prueba_2025_v2.parquet"
ruta_diccionario = carpeta_modelado / "metadata" / "diccionario_variables_modelado_v2.csv"
ruta_configuracion = carpeta_modelado / "metadata" / "configuracion_modelado_v2.csv"

rutas_entrada = {
    "Entrenamiento": ruta_entrenamiento,
    "Prueba": ruta_prueba,
    "Diccionario": ruta_diccionario,
    "Configuración": ruta_configuracion,
}

archivos_faltantes = [str(ruta) for ruta in rutas_entrada.values() if not ruta.is_file()]

if archivos_faltantes:
    raise FileNotFoundError("No se encontraron los siguientes archivos:\n" + "\n".join(archivos_faltantes))

pd.DataFrame(
    {
        "archivo": rutas_entrada.keys(),
        "ruta": [str(ruta.relative_to(raiz_proyecto)) for ruta in rutas_entrada.values()],
        "existe": [ruta.is_file() for ruta in rutas_entrada.values()],
    }
)

,archivo,ruta,existe
0,Entrenamiento,data/processed/geminidas_modelado_v2/geminidas...,True
1,Prueba,data/processed/geminidas_modelado_v2/geminidas...,True
2,Diccionario,data/processed/geminidas_modelado_v2/metadata/...,True
3,Configuración,data/processed/geminidas_modelado_v2/metadata/...,True


In [3]:
# Cargar las bases y auditar su estructura, documentación y separación temporal

df_entrenamiento = pd.read_parquet(ruta_entrenamiento)
df_prueba = pd.read_parquet(ruta_prueba)
df_diccionario = pd.read_csv(ruta_diccionario)
df_configuracion = pd.read_csv(ruta_configuracion)

variables_documentadas = set(df_diccionario["variable"])
columnas_entrenamiento = set(df_entrenamiento.columns)
columnas_prueba = set(df_prueba.columns)
identificadores_entrenamiento = set(df_entrenamiento["id_fila_origen"])
identificadores_prueba = set(df_prueba["id_fila_origen"])

resumen_bases = pd.DataFrame(
    [
        {
            "base": "Entrenamiento",
            "filas": len(df_entrenamiento),
            "columnas": df_entrenamiento.shape[1],
            "anio_minimo": df_entrenamiento["source_year"].min(),
            "anio_maximo": df_entrenamiento["source_year"].max(),
            "identificadores_unicos": df_entrenamiento["id_fila_origen"].is_unique,
        },
        {
            "base": "Prueba",
            "filas": len(df_prueba),
            "columnas": df_prueba.shape[1],
            "anio_minimo": df_prueba["source_year"].min(),
            "anio_maximo": df_prueba["source_year"].max(),
            "identificadores_unicos": df_prueba["id_fila_origen"].is_unique,
        },
    ]
)

comprobaciones_estructura = pd.DataFrame(
    [
        {"comprobacion": "Filas de entrenamiento correctas", "resultado": len(df_entrenamiento) == 12904},
        {"comprobacion": "Filas de prueba correctas", "resultado": len(df_prueba) == 6175},
        {"comprobacion": "Ambas bases tienen las mismas columnas", "resultado": columnas_entrenamiento == columnas_prueba},
        {"comprobacion": "Todas las columnas están documentadas", "resultado": columnas_entrenamiento == variables_documentadas},
        {"comprobacion": "Identificadores de entrenamiento únicos", "resultado": df_entrenamiento["id_fila_origen"].is_unique},
        {"comprobacion": "Identificadores de prueba únicos", "resultado": df_prueba["id_fila_origen"].is_unique},
        {"comprobacion": "Sin identificadores compartidos", "resultado": identificadores_entrenamiento.isdisjoint(identificadores_prueba)},
        {"comprobacion": "Entrenamiento contiene 2019–2024", "resultado": set(df_entrenamiento["source_year"].unique()) == set(range(2019, 2025))},
        {"comprobacion": "Prueba contiene solamente 2025", "resultado": set(df_prueba["source_year"].unique()) == {2025}},
        {"comprobacion": "Partición de entrenamiento correcta", "resultado": set(df_entrenamiento["particion_modelado"].unique()) == {"entrenamiento"}},
        {"comprobacion": "Partición de prueba correcta", "resultado": set(df_prueba["particion_modelado"].unique()) == {"prueba"}},
    ]
)

display(resumen_bases)
display(comprobaciones_estructura)

if not comprobaciones_estructura["resultado"].all():
    raise ValueError("La auditoría estructural detectó al menos una inconsistencia.")

,base,filas,columnas,anio_minimo,anio_maximo,identificadores_unicos
0,Entrenamiento,12904,39,2019,2024,True
1,Prueba,6175,39,2025,2025,True


,comprobacion,resultado
0,Filas de entrenamiento correctas,True
1,Filas de prueba correctas,True
2,Ambas bases tienen las mismas columnas,True
3,Todas las columnas están documentadas,True
4,Identificadores de entrenamiento únicos,True
5,Identificadores de prueba únicos,True
6,Sin identificadores compartidos,True
7,Entrenamiento contiene 2019–2024,True
8,Prueba contiene solamente 2025,True
9,Partición de entrenamiento correcta,True


### Construcción de las muestras utilizables para el modelo

In [4]:
# Construir las muestras estándar y estricta utilizables para el modelado

columnas_seleccion_ml = ["id_fila_origen", "usar_modelo_ml", "objetivo_ml", "grupo_brillo_ml", "pertenece_muestra_estricta"]
columnas_ausentes_ml = [columna for columna in columnas_seleccion_ml if columna not in df_entrenamiento.columns]

if columnas_ausentes_ml:
    raise KeyError(f"Faltan columnas necesarias para construir las muestras de ML: {columnas_ausentes_ml}")

df_entrenamiento_ml = df_entrenamiento.loc[df_entrenamiento["usar_modelo_ml"]].copy()
df_prueba_ml = df_prueba.loc[df_prueba["usar_modelo_ml"]].copy()
df_entrenamiento_ml_estricto = df_entrenamiento_ml.loc[df_entrenamiento_ml["pertenece_muestra_estricta"]].copy()
df_prueba_ml_estricto = df_prueba_ml.loc[df_prueba_ml["pertenece_muestra_estricta"]].copy()

resumen_muestras_ml = pd.DataFrame(
    [
        {
            "particion": "Entrenamiento",
            "muestra": "Estándar",
            "trayectorias_base": len(df_entrenamiento),
            "trayectorias_utilizables_ml": len(df_entrenamiento_ml),
            "porcentaje_utilizable": 100 * len(df_entrenamiento_ml) / len(df_entrenamiento),
            "objetivos_faltantes": df_entrenamiento_ml["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Entrenamiento",
            "muestra": "Estricta",
            "trayectorias_base": df_entrenamiento["pertenece_muestra_estricta"].sum(),
            "trayectorias_utilizables_ml": len(df_entrenamiento_ml_estricto),
            "porcentaje_utilizable": 100 * len(df_entrenamiento_ml_estricto) / df_entrenamiento["pertenece_muestra_estricta"].sum(),
            "objetivos_faltantes": df_entrenamiento_ml_estricto["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Prueba",
            "muestra": "Estándar",
            "trayectorias_base": len(df_prueba),
            "trayectorias_utilizables_ml": len(df_prueba_ml),
            "porcentaje_utilizable": 100 * len(df_prueba_ml) / len(df_prueba),
            "objetivos_faltantes": df_prueba_ml["objetivo_ml"].isna().sum(),
        },
        {
            "particion": "Prueba",
            "muestra": "Estricta",
            "trayectorias_base": df_prueba["pertenece_muestra_estricta"].sum(),
            "trayectorias_utilizables_ml": len(df_prueba_ml_estricto),
            "porcentaje_utilizable": 100 * len(df_prueba_ml_estricto) / df_prueba["pertenece_muestra_estricta"].sum(),
            "objetivos_faltantes": df_prueba_ml_estricto["objetivo_ml"].isna().sum(),
        },
    ]
)

comprobaciones_muestras_ml = pd.DataFrame(
    [
        {"comprobacion": "Entrenamiento estándar conserva 8,632 casos", "resultado": len(df_entrenamiento_ml) == 8632},
        {"comprobacion": "Entrenamiento estricto conserva 2,727 casos", "resultado": len(df_entrenamiento_ml_estricto) == 2727},
        {"comprobacion": "Prueba estándar conserva 4,182 casos", "resultado": len(df_prueba_ml) == 4182},
        {"comprobacion": "Prueba estricta conserva 1,943 casos", "resultado": len(df_prueba_ml_estricto) == 1943},
        {"comprobacion": "Objetivo completo en entrenamiento", "resultado": df_entrenamiento_ml["objetivo_ml"].notna().all()},
        {"comprobacion": "Objetivo binario en entrenamiento", "resultado": set(df_entrenamiento_ml["objetivo_ml"].unique()) == {0, 1}},
        {"comprobacion": "Identificadores únicos en entrenamiento ML", "resultado": df_entrenamiento_ml["id_fila_origen"].is_unique},
        {"comprobacion": "Identificadores únicos en prueba ML", "resultado": df_prueba_ml["id_fila_origen"].is_unique},
    ]
)

display(resumen_muestras_ml.round({"porcentaje_utilizable": 2}))
display(comprobaciones_muestras_ml)

if not comprobaciones_muestras_ml["resultado"].all():
    raise ValueError("La construcción de las muestras de modelado detectó alguna inconsistencia.")

,particion,muestra,trayectorias_base,trayectorias_utilizables_ml,porcentaje_utilizable,objetivos_faltantes
0,Entrenamiento,Estándar,12904,8632,66.89,0
1,Entrenamiento,Estricta,4166,2727,65.46,0
2,Prueba,Estándar,6175,4182,67.72,0
3,Prueba,Estricta,2874,1943,67.61,0


,comprobacion,resultado
0,"Entrenamiento estándar conserva 8,632 casos",True
1,"Entrenamiento estricto conserva 2,727 casos",True
2,"Prueba estándar conserva 4,182 casos",True
3,"Prueba estricta conserva 1,943 casos",True
4,Objetivo completo en entrenamiento,True
5,Objetivo binario en entrenamiento,True
6,Identificadores únicos en entrenamiento ML,True
7,Identificadores únicos en prueba ML,True


### Definición y auditoría del objetivo binario

In [5]:
# Auditar la codificación, los límites y el balance del objetivo binario de entrenamiento

configuracion_modelado = df_configuracion.set_index("elemento")["valor"]
limite_brillante = float(configuracion_modelado.loc["Límite brillante"])
limite_tenue = float(configuracion_modelado.loc["Límite tenue"])

y_entrenamiento = df_entrenamiento_ml["objetivo_ml"].astype(int)
y_entrenamiento_estricto = df_entrenamiento_ml_estricto["objetivo_ml"].astype(int)

resumen_objetivo = (
    df_entrenamiento_ml.groupby("objetivo_ml", observed=True)
    .agg(
        trayectorias=("id_fila_origen", "size"),
        magnitud_minima=("magnitud_analisis", "min"),
        magnitud_mediana=("magnitud_analisis", "median"),
        magnitud_maxima=("magnitud_analisis", "max"),
    )
    .reset_index()
)

resumen_objetivo["clase"] = resumen_objetivo["objetivo_ml"].map({0: "Brillante", 1: "Tenue"})
resumen_objetivo["porcentaje_entrenamiento"] = 100 * resumen_objetivo["trayectorias"] / len(df_entrenamiento_ml)
resumen_objetivo = resumen_objetivo[["objetivo_ml", "clase", "trayectorias", "porcentaje_entrenamiento", "magnitud_minima", "magnitud_mediana", "magnitud_maxima"]]

conteos_estandar = y_entrenamiento.value_counts()
conteos_estrictos = y_entrenamiento_estricto.value_counts()

comprobaciones_objetivo = pd.DataFrame(
    [
        {"comprobacion": "Codificación binaria completa", "resultado": set(y_entrenamiento.unique()) == {0, 1}},
        {"comprobacion": "Clase brillante respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 0, "magnitud_analisis"].le(limite_brillante).all()},
        {"comprobacion": "Clase tenue respeta el límite", "resultado": df_entrenamiento_ml.loc[y_entrenamiento == 1, "magnitud_analisis"].ge(limite_tenue).all()},
        {"comprobacion": "Estándar contiene 4,300 brillantes", "resultado": conteos_estandar.get(0, 0) == 4300},
        {"comprobacion": "Estándar contiene 4,332 tenues", "resultado": conteos_estandar.get(1, 0) == 4332},
        {"comprobacion": "Estricta contiene 1,681 brillantes", "resultado": conteos_estrictos.get(0, 0) == 1681},
        {"comprobacion": "Estricta contiene 1,046 tenues", "resultado": conteos_estrictos.get(1, 0) == 1046},
    ]
)

print(f"Límite brillante: {limite_brillante:.2f}")
print(f"Límite tenue: {limite_tenue:.2f}")
display(resumen_objetivo.round(4))
display(comprobaciones_objetivo)

if not comprobaciones_objetivo["resultado"].all():
    raise ValueError("La auditoría del objetivo binario detectó alguna inconsistencia.")

Límite brillante: -0.89
Límite tenue: -0.07


,objetivo_ml,clase,trayectorias,porcentaje_entrenamiento,magnitud_minima,magnitud_mediana,magnitud_maxima
0,0,Brillante,4300,49.8146,-7.64,-1.38,-0.89
1,1,Tenue,4332,50.1854,-0.07,0.44,3.82


,comprobacion,resultado
0,Codificación binaria completa,True
1,Clase brillante respeta el límite,True
2,Clase tenue respeta el límite,True
3,"Estándar contiene 4,300 brillantes",True
4,"Estándar contiene 4,332 tenues",True
5,"Estricta contiene 1,681 brillantes",True
6,"Estricta contiene 1,046 tenues",True


### Definición y auditoría de los bloques de predictores

In [6]:
# Definir los bloques de predictores y comprobar su integridad numérica

variables_fase = ["delta_sol_lon_deg"]
variables_calidad = ["Qc (deg)", "MedianFitErr (arcsec)", "numero_estaciones", "sigma_radiante_aprox", "Vgeo_sigma_modelo"]
variables_cinematica = ["delta_v_paralela_km_s", "delta_v_lambda_km_s", "delta_v_beta_km_s"]
variables_validacion = ["source_year"]
variables_pca_futuras = ["PC1", "PC2", "PC3"]

bloques_predictores = {
    "Fase solar": variables_fase,
    "Calidad observacional": variables_calidad,
    "Cinemática residual local": variables_cinematica,
}

variables_requeridas = variables_fase + variables_calidad + variables_cinematica + variables_validacion
variables_ausentes = [variable for variable in variables_requeridas if variable not in df_entrenamiento_ml.columns]

if variables_ausentes:
    raise KeyError(f"Faltan variables necesarias para el modelado: {variables_ausentes}")

filas_resumen_entrenamiento = []

for bloque, variables in bloques_predictores.items():
    for variable in variables:
        valores = pd.to_numeric(df_entrenamiento_ml[variable], errors="coerce")
        filas_resumen_entrenamiento.append(
            {
                "bloque": bloque,
                "variable": variable,
                "faltantes": valores.isna().sum(),
                "no_finitos": (~np.isfinite(valores.dropna())).sum(),
                "minimo": valores.min(),
                "mediana": valores.median(),
                "maximo": valores.max(),
            }
        )

resumen_predictores_entrenamiento = pd.DataFrame(filas_resumen_entrenamiento)

auditoria_particiones = pd.DataFrame(
    [
        {
            "particion": nombre,
            "trayectorias": len(base),
            "faltantes_predictores": base[variables_requeridas].isna().sum().sum(),
            "valores_no_finitos": sum((~np.isfinite(pd.to_numeric(base[variable], errors="coerce").dropna())).sum() for variable in variables_requeridas),
        }
        for nombre, base in [("Entrenamiento", df_entrenamiento_ml), ("Prueba", df_prueba_ml)]
    ]
)

display(resumen_predictores_entrenamiento.round(6))
display(auditoria_particiones)

if resumen_predictores_entrenamiento[["faltantes", "no_finitos"]].to_numpy().sum() != 0:
    raise ValueError("Los predictores de entrenamiento contienen valores faltantes o no finitos.")

if auditoria_particiones[["faltantes_predictores", "valores_no_finitos"]].to_numpy().sum() != 0:
    raise ValueError("Alguna partición contiene predictores faltantes o no finitos.")

,bloque,variable,faltantes,no_finitos,minimo,mediana,maximo
0,Fase solar,delta_sol_lon_deg,0,0,-4.990121,-0.307996,1.983771
1,Calidad observacional,Qc (deg),0,0,15.080000,65.935000,90.000000
2,Calidad observacional,MedianFitErr (arcsec),0,0,2.730000,43.565000,60.000000
3,Calidad observacional,numero_estaciones,0,0,2.000000,3.000000,20.000000
4,Calidad observacional,sigma_radiante_aprox,0,0,0.009896,0.161504,0.249968
5,Calidad observacional,Vgeo_sigma_modelo,0,0,0.000500,0.060200,2.130500
6,Cinemática residual local,delta_v_paralela_km_s,0,0,-3.478542,-0.008750,3.471391
7,Cinemática residual local,delta_v_lambda_km_s,0,0,-1.245482,0.010680,1.192395
8,Cinemática residual local,delta_v_beta_km_s,0,0,-1.179222,0.007675,1.096174


,particion,trayectorias,faltantes_predictores,valores_no_finitos
0,Entrenamiento,8632,0,0
1,Prueba,4182,0,0


### Diseño de la validación temporal interna

In [7]:
# Construir y auditar los pliegues de validación temporal creciente

def construir_pliegues_temporales(df_modelado, nombre_muestra):
    anios = sorted(df_modelado["source_year"].unique())
    pliegues = []
    filas_resumen = []

    for anio_validacion in anios[1:]:
        mascara_entrenamiento = df_modelado["source_year"] < anio_validacion
        mascara_validacion = df_modelado["source_year"] == anio_validacion
        indices_entrenamiento = np.flatnonzero(mascara_entrenamiento.to_numpy())
        indices_validacion = np.flatnonzero(mascara_validacion.to_numpy())
        objetivo_entrenamiento = df_modelado.iloc[indices_entrenamiento]["objetivo_ml"].astype(int)
        objetivo_validacion = df_modelado.iloc[indices_validacion]["objetivo_ml"].astype(int)

        pliegues.append((indices_entrenamiento, indices_validacion))
        filas_resumen.append(
            {
                "muestra": nombre_muestra,
                "pliegue": len(pliegues),
                "anios_entrenamiento": f"{df_modelado.iloc[indices_entrenamiento]['source_year'].min()}–{df_modelado.iloc[indices_entrenamiento]['source_year'].max()}",
                "anio_validacion": anio_validacion,
                "entrenamiento_total": len(indices_entrenamiento),
                "entrenamiento_brillantes": (objetivo_entrenamiento == 0).sum(),
                "entrenamiento_tenues": (objetivo_entrenamiento == 1).sum(),
                "validacion_total": len(indices_validacion),
                "validacion_brillantes": (objetivo_validacion == 0).sum(),
                "validacion_tenues": (objetivo_validacion == 1).sum(),
            }
        )

    return pliegues, pd.DataFrame(filas_resumen)

pliegues_temporales_estandar, resumen_pliegues_estandar = construir_pliegues_temporales(df_entrenamiento_ml, "Estándar")
pliegues_temporales_estrictos, resumen_pliegues_estrictos = construir_pliegues_temporales(df_entrenamiento_ml_estricto, "Estricta")
resumen_pliegues_temporales = pd.concat([resumen_pliegues_estandar, resumen_pliegues_estrictos], ignore_index=True)

comprobaciones_pliegues = pd.DataFrame(
    [
        {"comprobacion": "Cinco pliegues estándar", "resultado": len(pliegues_temporales_estandar) == 5},
        {"comprobacion": "Cinco pliegues estrictos", "resultado": len(pliegues_temporales_estrictos) == 5},
        {"comprobacion": "Ambas clases presentes en cada entrenamiento", "resultado": (resumen_pliegues_temporales[["entrenamiento_brillantes", "entrenamiento_tenues"]] > 0).all().all()},
        {"comprobacion": "Ambas clases presentes en cada validación", "resultado": (resumen_pliegues_temporales[["validacion_brillantes", "validacion_tenues"]] > 0).all().all()},
        {"comprobacion": "2025 permanece fuera de la validación interna", "resultado": resumen_pliegues_temporales["anio_validacion"].max() == 2024},
    ]
)

display(resumen_pliegues_temporales)
display(comprobaciones_pliegues)

if not comprobaciones_pliegues["resultado"].all():
    raise ValueError("El esquema de validación temporal no satisface todas las condiciones.")

,muestra,pliegue,anios_entrenamiento,anio_validacion,entrenamiento_total,entrenamiento_brillantes,entrenamiento_tenues,validacion_total,validacion_brillantes,validacion_tenues
0,Estándar,1,2019–2019,2020,274,131,143,703,319,384
1,Estándar,2,2019–2020,2021,977,450,527,1319,437,882
2,Estándar,3,2019–2021,2022,2296,887,1409,2433,1405,1028
3,Estándar,4,2019–2022,2023,4729,2292,2437,2258,997,1261
4,Estándar,5,2019–2023,2024,6987,3289,3698,1645,1011,634
5,Estricta,1,2019–2019,2020,58,39,19,116,56,60
6,Estricta,2,2019–2020,2021,174,95,79,192,61,131
7,Estricta,3,2019–2021,2022,366,156,210,1153,799,354
8,Estricta,4,2019–2022,2023,1519,955,564,824,423,401
9,Estricta,5,2019–2023,2024,2343,1378,965,384,303,81


,comprobacion,resultado
0,Cinco pliegues estándar,True
1,Cinco pliegues estrictos,True
2,Ambas clases presentes en cada entrenamiento,True
3,Ambas clases presentes en cada validación,True
4,2025 permanece fuera de la validación interna,True


### Definición del preprocesamiento dentro de cada pliegue

In [9]:
# Definir el preprocesamiento de fase, calidad y cinemática sin ajustarlo todavía

transformador_fase = Pipeline(
    [
        ("terminos_fase", PolynomialFeatures(degree=2, include_bias=False)),
        ("escalamiento_fase", StandardScaler()),
    ]
)

transformador_calidad = Pipeline(
    [
        ("escalamiento_calidad", StandardScaler()),
    ]
)

transformador_cinematica = Pipeline(
    [
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocesador_control_cinematica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

configuracion_preprocesamiento = pd.DataFrame(
    [
        {
            "bloque": "Fase solar",
            "variables_originales": ", ".join(variables_fase),
            "transformacion": "Término lineal + término cuadrático + estandarización",
            "numero_salidas": 2,
        },
        {
            "bloque": "Calidad observacional",
            "variables_originales": ", ".join(variables_calidad),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_calidad),
        },
        {
            "bloque": "Cinemática residual local",
            "variables_originales": ", ".join(variables_cinematica),
            "transformacion": "Estandarización",
            "numero_salidas": len(variables_cinematica),
        },
    ]
)

configuracion_modelos_iniciales = pd.DataFrame(
    [
        {
            "modelo": "Control",
            "fase": True,
            "calidad": True,
            "cinematica": False,
            "numero_predictores_transformados": 2 + len(variables_calidad),
        },
        {
            "modelo": "Control + cinemática local",
            "fase": True,
            "calidad": True,
            "cinematica": True,
            "numero_predictores_transformados": 2 + len(variables_calidad) + len(variables_cinematica),
        },
    ]
)

display(configuracion_preprocesamiento)
display(configuracion_modelos_iniciales)

if hasattr(preprocesador_control, "transformers_") or hasattr(preprocesador_control_cinematica, "transformers_"):
    raise RuntimeError("Los preprocesadores no deben estar ajustados antes de la validación temporal.")

,bloque,variables_originales,transformacion,numero_salidas
0,Fase solar,delta_sol_lon_deg,Término lineal + término cuadrático + estandar...,2
1,Calidad observacional,"Qc (deg), MedianFitErr (arcsec), numero_estaci...",Estandarización,5
2,Cinemática residual local,"delta_v_paralela_km_s, delta_v_lambda_km_s, de...",Estandarización,3


,modelo,fase,calidad,cinematica,numero_predictores_transformados
0,Control,True,True,False,7
1,Control + cinemática local,True,True,True,10


## Etapa 12. Entrenamiento y comparación temporal de los modelos

### Modelo de control con fase solar y calidad observacional

In [12]:
# Entrenar y evaluar temporalmente el modelo de control mediante regresión logística

def evaluar_modelo_temporal(modelo, nombre_modelo, df_modelado, pliegues):
    filas_metricas = []
    tablas_predicciones = []

    for numero_pliegue, (indices_entrenamiento, indices_validacion) in enumerate(pliegues, start=1):
        modelo_pliegue = clone(modelo)
        X_entrenamiento = df_modelado.iloc[indices_entrenamiento]
        y_entrenamiento_pliegue = X_entrenamiento["objetivo_ml"].astype(int)
        X_validacion = df_modelado.iloc[indices_validacion]
        y_validacion = X_validacion["objetivo_ml"].astype(int)
        anio_validacion = int(X_validacion["source_year"].iloc[0])

        modelo_pliegue.fit(X_entrenamiento, y_entrenamiento_pliegue)

        probabilidades = modelo_pliegue.predict_proba(X_validacion)[:, 1]
        predicciones = (probabilidades >= 0.5).astype(int)

        tabla_predicciones = pd.DataFrame(
            {
                "id_fila_origen": X_validacion["id_fila_origen"].to_numpy(),
                "modelo": nombre_modelo,
                "pliegue": numero_pliegue,
                "anio_validacion": anio_validacion,
                "objetivo_real": y_validacion.to_numpy(),
                "probabilidad_tenue": probabilidades,
                "prediccion": predicciones,
                "pertenece_muestra_estricta": X_validacion["pertenece_muestra_estricta"].to_numpy(),
            }
        )

        tablas_predicciones.append(tabla_predicciones)

        for nombre_muestra, mascara in [
            ("Estándar", np.ones(len(X_validacion), dtype=bool)),
            ("Estricta", X_validacion["pertenece_muestra_estricta"].to_numpy(dtype=bool)),
        ]:
            y_real_muestra = y_validacion.to_numpy()[mascara]
            probabilidades_muestra = probabilidades[mascara]
            predicciones_muestra = predicciones[mascara]
            verdaderos_brillantes, falsos_tenues, falsos_brillantes, verdaderos_tenues = confusion_matrix(y_real_muestra, predicciones_muestra, labels=[0, 1]).ravel()

            filas_metricas.append(
                {
                    "modelo": nombre_modelo,
                    "muestra": nombre_muestra,
                    "pliegue": numero_pliegue,
                    "anio_validacion": anio_validacion,
                    "trayectorias": len(y_real_muestra),
                    "brillantes": (y_real_muestra == 0).sum(),
                    "tenues": (y_real_muestra == 1).sum(),
                    "roc_auc": roc_auc_score(y_real_muestra, probabilidades_muestra),
                    "pr_auc": average_precision_score(y_real_muestra, probabilidades_muestra),
                    "exactitud": accuracy_score(y_real_muestra, predicciones_muestra),
                    "exactitud_balanceada": balanced_accuracy_score(y_real_muestra, predicciones_muestra),
                    "sensibilidad_tenues": recall_score(y_real_muestra, predicciones_muestra, pos_label=1),
                    "especificidad_brillantes": recall_score(y_real_muestra, predicciones_muestra, pos_label=0),
                    "verdaderos_brillantes": verdaderos_brillantes,
                    "falsos_tenues": falsos_tenues,
                    "falsos_brillantes": falsos_brillantes,
                    "verdaderos_tenues": verdaderos_tenues,
                }
            )

    return pd.DataFrame(filas_metricas), pd.concat(tablas_predicciones, ignore_index=True)

modelo_control = Pipeline(
    [
        ("preprocesamiento", preprocesador_control),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_control, predicciones_control = evaluar_modelo_temporal(modelo_control, "Control", df_entrenamiento_ml, pliegues_temporales_estandar)

columnas_metricas = ["roc_auc", "pr_auc", "exactitud", "exactitud_balanceada", "sensibilidad_tenues", "especificidad_brillantes"]
resumen_control = metricas_control.groupby(["modelo", "muestra"])[columnas_metricas].agg(["mean", "std"]).reset_index()
resumen_control.columns = ["modelo", "muestra"] + [f"{metrica}_{estadistico}" for metrica, estadistico in resumen_control.columns.tolist()[2:]]

display(metricas_control.round(4))
display(resumen_control.round(4))

if len(predicciones_control) != sum(len(indices_validacion) for _, indices_validacion in pliegues_temporales_estandar):
    raise ValueError("No se conservaron todas las predicciones temporales esperadas.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,pr_auc,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control,Estándar,1,2020,703,319,384,0.7496,0.7995,0.6728,0.6666,0.7344,0.5987,191,128,102,282
1,Control,Estricta,1,2020,116,56,60,0.8104,0.7867,0.6293,0.6393,0.3500,0.9286,52,4,39,21
2,Control,Estándar,2,2021,1319,437,882,0.8172,0.8904,0.7612,0.7296,0.8231,0.6362,278,159,156,726
3,Control,Estricta,2,2021,192,61,131,0.7686,0.8618,0.6927,0.7266,0.6336,0.8197,50,11,48,83
4,Control,Estándar,3,2022,2433,1405,1028,0.6726,0.6200,0.6108,0.6224,0.6975,0.5473,769,636,311,717
5,Control,Estricta,3,2022,1153,799,354,0.6124,0.4440,0.5716,0.5768,0.5904,0.5632,450,349,145,209
6,Control,Estándar,4,2023,2258,997,1261,0.7426,0.7741,0.6634,0.6754,0.5726,0.7783,776,221,539,722
7,Control,Estricta,4,2023,824,423,401,0.7452,0.7373,0.6383,0.6307,0.3441,0.9173,388,35,263,138
8,Control,Estándar,5,2024,1645,1011,634,0.7918,0.7238,0.7052,0.7157,0.7618,0.6696,677,334,151,483
9,Control,Estricta,5,2024,384,303,81,0.6859,0.4186,0.7734,0.6393,0.4074,0.8713,264,39,48,33


,modelo,muestra,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,exactitud_mean,exactitud_std,exactitud_balanceada_mean,exactitud_balanceada_std,sensibilidad_tenues_mean,sensibilidad_tenues_std,especificidad_brillantes_mean,especificidad_brillantes_std
0,Control,Estricta,0.7245,0.0771,0.6497,0.2044,0.6611,0.0761,0.6426,0.0537,0.4651,0.1372,0.820,0.1499
1,Control,Estándar,0.7548,0.0552,0.7616,0.0996,0.6827,0.0555,0.6820,0.0425,0.7179,0.0933,0.646,0.0868


### 12.2 Modelo con componentes cinemáticas locales lineales

#### 12.2.1 Versión lineal

In [15]:
# Entrenar el modelo cinemático lineal y comparar su desempeño con el modelo de control

modelo_cinematica_lineal = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_lineal, predicciones_cinematica_lineal = evaluar_modelo_temporal(
    modelo_cinematica_lineal,
    "Control + cinemática lineal",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

comparacion_control_cinematica = metricas_control.merge(
    metricas_cinematica_lineal,
    on=["muestra", "pliegue", "anio_validacion"],
    suffixes=("_control", "_cinematica"),
    validate="one_to_one",
)

for metrica in columnas_metricas:
    comparacion_control_cinematica[f"delta_{metrica}"] = comparacion_control_cinematica[f"{metrica}_cinematica"] - comparacion_control_cinematica[f"{metrica}_control"]

filas_resumen_mejoras = []

for muestra, grupo in comparacion_control_cinematica.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in columnas_metricas:
        diferencias = grupo[f"delta_{metrica}"]
        fila[f"delta_medio_{metrica}"] = diferencias.mean()
        fila[f"desviacion_delta_{metrica}"] = diferencias.std()
        fila[f"pliegues_con_mejora_{metrica}"] = (diferencias > 0).sum()

    filas_resumen_mejoras.append(fila)

resumen_mejoras_cinematica = pd.DataFrame(filas_resumen_mejoras)
columnas_comparacion_anual = ["muestra", "anio_validacion"] + [f"delta_{metrica}" for metrica in columnas_metricas]

display(metricas_cinematica_lineal.round(4))
display(comparacion_control_cinematica[columnas_comparacion_anual].round(4))
display(resumen_mejoras_cinematica.round(4))

if len(metricas_cinematica_lineal) != len(metricas_control):
    raise ValueError("El modelo cinemático y el modelo de control no produjeron el mismo número de evaluaciones.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,pr_auc,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control + cinemática lineal,Estándar,1,2020,703,319,384,0.7709,0.8241,0.6942,0.6890,0.7448,0.6332,202,117,98,286
1,Control + cinemática lineal,Estricta,1,2020,116,56,60,0.8247,0.8064,0.6638,0.6714,0.4500,0.8929,50,6,33,27
2,Control + cinemática lineal,Estándar,2,2021,1319,437,882,0.8153,0.9000,0.7604,0.7337,0.8129,0.6545,286,151,165,717
3,Control + cinemática lineal,Estricta,2,2021,192,61,131,0.7600,0.8614,0.6667,0.7032,0.6031,0.8033,49,12,52,79
4,Control + cinemática lineal,Estándar,3,2022,2433,1405,1028,0.7088,0.6591,0.6449,0.6495,0.6790,0.6199,871,534,330,698
5,Control + cinemática lineal,Estricta,3,2022,1153,799,354,0.6653,0.5095,0.6357,0.6184,0.5734,0.6633,530,269,151,203
6,Control + cinemática lineal,Estándar,4,2023,2258,997,1261,0.7639,0.7919,0.6780,0.6915,0.5765,0.8064,804,193,534,727
7,Control + cinemática lineal,Estricta,4,2023,824,423,401,0.7879,0.7896,0.6626,0.6553,0.3815,0.9291,393,30,248,153
8,Control + cinemática lineal,Estándar,5,2024,1645,1011,634,0.8007,0.7373,0.7155,0.7244,0.7634,0.6855,693,318,150,484
9,Control + cinemática lineal,Estricta,5,2024,384,303,81,0.7326,0.4732,0.7656,0.6615,0.4815,0.8416,255,48,42,39


,muestra,anio_validacion,delta_roc_auc,delta_pr_auc,delta_exactitud,delta_exactitud_balanceada,delta_sensibilidad_tenues,delta_especificidad_brillantes
0,Estándar,2020,0.0213,0.0246,0.0213,0.0224,0.0104,0.0345
1,Estricta,2020,0.0143,0.0197,0.0345,0.0321,0.1000,-0.0357
2,Estándar,2021,-0.0019,0.0096,-0.0008,0.0041,-0.0102,0.0183
3,Estricta,2021,-0.0086,-0.0004,-0.0260,-0.0235,-0.0305,-0.0164
4,Estándar,2022,0.0362,0.0391,0.0341,0.0271,-0.0185,0.0726
5,Estricta,2022,0.0529,0.0655,0.0642,0.0416,-0.0169,0.1001
6,Estándar,2023,0.0213,0.0178,0.0146,0.0160,0.0040,0.0281
7,Estricta,2023,0.0427,0.0523,0.0243,0.0246,0.0374,0.0118
8,Estándar,2024,0.0089,0.0135,0.0103,0.0087,0.0016,0.0158
9,Estricta,2024,0.0467,0.0546,-0.0078,0.0222,0.0741,-0.0297


,muestra,delta_medio_roc_auc,desviacion_delta_roc_auc,pliegues_con_mejora_roc_auc,delta_medio_pr_auc,desviacion_delta_pr_auc,pliegues_con_mejora_pr_auc,delta_medio_exactitud,desviacion_delta_exactitud,pliegues_con_mejora_exactitud,delta_medio_exactitud_balanceada,desviacion_delta_exactitud_balanceada,pliegues_con_mejora_exactitud_balanceada,delta_medio_sensibilidad_tenues,desviacion_delta_sensibilidad_tenues,pliegues_con_mejora_sensibilidad_tenues,delta_medio_especificidad_brillantes,desviacion_delta_especificidad_brillantes,pliegues_con_mejora_especificidad_brillantes
0,Estricta,0.0296,0.0260,4,0.0384,0.0276,4,0.0178,0.0355,3,0.0194,0.0251,4,0.0328,0.0564,3,0.0060,0.0557,2
1,Estándar,0.0172,0.0144,4,0.0209,0.0116,5,0.0159,0.0130,4,0.0157,0.0095,5,-0.0025,0.0116,3,0.0339,0.0229,5


#### 12.2.2 Versión cuadrática

In [17]:
# Entrenar el modelo cinemático cuadrático y compararlo con los modelos anteriores

def agregar_cuadrados_cinematicos(X):
    valores = np.asarray(X, dtype=float)
    return np.column_stack([valores, valores**2])

transformador_cinematica_cuadratica = Pipeline(
    [
        ("terminos_cinematicos", FunctionTransformer(agregar_cuadrados_cinematicos, validate=False)),
        ("escalamiento_cinematica", StandardScaler()),
    ]
)

preprocesador_control_cinematica_cuadratica = ColumnTransformer(
    [
        ("fase", transformador_fase, variables_fase),
        ("calidad", transformador_calidad, variables_calidad),
        ("cinematica", transformador_cinematica_cuadratica, variables_cinematica),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

modelo_cinematica_cuadratica = Pipeline(
    [
        ("preprocesamiento", preprocesador_control_cinematica_cuadratica),
        ("clasificador", LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000)),
    ]
)

metricas_cinematica_cuadratica, predicciones_cinematica_cuadratica = evaluar_modelo_temporal(
    modelo_cinematica_cuadratica,
    "Control + cinemática cuadrática",
    df_entrenamiento_ml,
    pliegues_temporales_estandar,
)

claves_comparacion = ["muestra", "pliegue", "anio_validacion"]
metricas_clave = ["roc_auc", "pr_auc", "exactitud_balanceada"]

comparacion_tres_modelos = (
    metricas_control[claves_comparacion + metricas_clave]
    .rename(columns={metrica: f"{metrica}_control" for metrica in metricas_clave})
    .merge(
        metricas_cinematica_lineal[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_lineal" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
    .merge(
        metricas_cinematica_cuadratica[claves_comparacion + metricas_clave].rename(columns={metrica: f"{metrica}_cuadratica" for metrica in metricas_clave}),
        on=claves_comparacion,
        validate="one_to_one",
    )
)

for metrica in metricas_clave:
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_control"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_control"]
    comparacion_tres_modelos[f"delta_{metrica}_cuadratica_vs_lineal"] = comparacion_tres_modelos[f"{metrica}_cuadratica"] - comparacion_tres_modelos[f"{metrica}_lineal"]

filas_resumen_cuadratico = []

for muestra, grupo in comparacion_tres_modelos.groupby("muestra"):
    fila = {"muestra": muestra}

    for metrica in metricas_clave:
        diferencia_control = grupo[f"delta_{metrica}_cuadratica_vs_control"]
        diferencia_lineal = grupo[f"delta_{metrica}_cuadratica_vs_lineal"]
        fila[f"delta_medio_{metrica}_vs_control"] = diferencia_control.mean()
        fila[f"pliegues_mejora_{metrica}_vs_control"] = (diferencia_control > 0).sum()
        fila[f"delta_medio_{metrica}_vs_lineal"] = diferencia_lineal.mean()
        fila[f"pliegues_mejora_{metrica}_vs_lineal"] = (diferencia_lineal > 0).sum()

    filas_resumen_cuadratico.append(fila)

resumen_modelo_cuadratico = pd.DataFrame(filas_resumen_cuadratico)

display(metricas_cinematica_cuadratica.round(4))
display(comparacion_tres_modelos.round(4))
display(resumen_modelo_cuadratico.round(4))

if len(metricas_cinematica_cuadratica) != len(metricas_control):
    raise ValueError("El modelo cuadrático no produjo el número esperado de evaluaciones.")

,modelo,muestra,pliegue,anio_validacion,trayectorias,brillantes,tenues,roc_auc,pr_auc,exactitud,exactitud_balanceada,sensibilidad_tenues,especificidad_brillantes,verdaderos_brillantes,falsos_tenues,falsos_brillantes,verdaderos_tenues
0,Control + cinemática cuadrática,Estándar,1,2020,703,319,384,0.7943,0.8372,0.7155,0.7083,0.7865,0.6301,201,118,82,302
1,Control + cinemática cuadrática,Estricta,1,2020,116,56,60,0.8283,0.8226,0.6638,0.6720,0.4333,0.9107,51,5,34,26
2,Control + cinemática cuadrática,Estándar,2,2021,1319,437,882,0.8277,0.9096,0.7665,0.7475,0.8039,0.6911,302,135,173,709
3,Control + cinemática cuadrática,Estricta,2,2021,192,61,131,0.7580,0.8685,0.6562,0.6912,0.5954,0.7869,48,13,53,78
4,Control + cinemática cuadrática,Estándar,3,2022,2433,1405,1028,0.7158,0.6728,0.6580,0.6625,0.6916,0.6335,890,515,317,711
5,Control + cinemática cuadrática,Estricta,3,2022,1153,799,354,0.6806,0.5354,0.6574,0.6474,0.6215,0.6733,538,261,134,220
6,Control + cinemática cuadrática,Estándar,4,2023,2258,997,1261,0.7740,0.7915,0.6842,0.6989,0.5734,0.8245,822,175,538,723
7,Control + cinemática cuadrática,Estricta,4,2023,824,423,401,0.7998,0.7940,0.6760,0.6688,0.4015,0.9362,396,27,240,161
8,Control + cinemática cuadrática,Estándar,5,2024,1645,1011,634,0.8105,0.7482,0.7277,0.7358,0.7713,0.7003,708,303,145,489
9,Control + cinemática cuadrática,Estricta,5,2024,384,303,81,0.7508,0.4997,0.7630,0.6599,0.4815,0.8383,254,49,42,39


,muestra,pliegue,anio_validacion,roc_auc_control,pr_auc_control,exactitud_balanceada_control,roc_auc_lineal,pr_auc_lineal,exactitud_balanceada_lineal,roc_auc_cuadratica,pr_auc_cuadratica,exactitud_balanceada_cuadratica,delta_roc_auc_cuadratica_vs_control,delta_roc_auc_cuadratica_vs_lineal,delta_pr_auc_cuadratica_vs_control,delta_pr_auc_cuadratica_vs_lineal,delta_exactitud_balanceada_cuadratica_vs_control,delta_exactitud_balanceada_cuadratica_vs_lineal
0,Estándar,1,2020,0.7496,0.7995,0.6666,0.7709,0.8241,0.6890,0.7943,0.8372,0.7083,0.0447,0.0234,0.0377,0.0131,0.0417,0.0193
1,Estricta,1,2020,0.8104,0.7867,0.6393,0.8247,0.8064,0.6714,0.8283,0.8226,0.6720,0.0179,0.0036,0.0358,0.0161,0.0327,0.0006
2,Estándar,2,2021,0.8172,0.8904,0.7296,0.8153,0.9000,0.7337,0.8277,0.9096,0.7475,0.0105,0.0124,0.0192,0.0096,0.0178,0.0138
3,Estricta,2,2021,0.7686,0.8618,0.7266,0.7600,0.8614,0.7032,0.7580,0.8685,0.6912,-0.0106,-0.0020,0.0067,0.0071,-0.0355,-0.0120
4,Estándar,3,2022,0.6726,0.6200,0.6224,0.7088,0.6591,0.6495,0.7158,0.6728,0.6625,0.0432,0.0070,0.0528,0.0137,0.0401,0.0131
5,Estricta,3,2022,0.6124,0.4440,0.5768,0.6653,0.5095,0.6184,0.6806,0.5354,0.6474,0.0682,0.0153,0.0914,0.0258,0.0706,0.0290
6,Estándar,4,2023,0.7426,0.7741,0.6754,0.7639,0.7919,0.6915,0.7740,0.7915,0.6989,0.0314,0.0101,0.0174,-0.0004,0.0235,0.0074
7,Estricta,4,2023,0.7452,0.7373,0.6307,0.7879,0.7896,0.6553,0.7998,0.7940,0.6688,0.0546,0.0119,0.0567,0.0044,0.0381,0.0135
8,Estándar,5,2024,0.7918,0.7238,0.7157,0.8007,0.7373,0.7244,0.8105,0.7482,0.7358,0.0187,0.0098,0.0244,0.0109,0.0201,0.0114
9,Estricta,5,2024,0.6859,0.4186,0.6393,0.7326,0.4732,0.6615,0.7508,0.4997,0.6599,0.0649,0.0183,0.0811,0.0265,0.0205,-0.0017


,muestra,delta_medio_roc_auc_vs_control,pliegues_mejora_roc_auc_vs_control,delta_medio_roc_auc_vs_lineal,pliegues_mejora_roc_auc_vs_lineal,delta_medio_pr_auc_vs_control,pliegues_mejora_pr_auc_vs_control,delta_medio_pr_auc_vs_lineal,pliegues_mejora_pr_auc_vs_lineal,delta_medio_exactitud_balanceada_vs_control,pliegues_mejora_exactitud_balanceada_vs_control,delta_medio_exactitud_balanceada_vs_lineal,pliegues_mejora_exactitud_balanceada_vs_lineal
0,Estricta,0.0390,4,0.0094,4,0.0544,5,0.0160,5,0.0253,4,0.0059,3
1,Estándar,0.0297,5,0.0125,5,0.0303,5,0.0094,4,0.0286,5,0.0130,5


### 12.3 Modelo PCA lineal